# 02 - Holt-Winters: Aditivo, Multiplicativo e Damped Trend - SOLUCAO

Este notebook explora o metodo de **Holt-Winters** usando a biblioteca **chronobox**, com **todos os exercicios resolvidos**.

**Conteudo:**
1. Holt-Winters aditivo vs multiplicativo
2. Damped trend (tendencia amortecida)
3. Comparacao no dataset airline.csv
4. Previsao e interpretacao dos parametros
5. **Exercicios resolvidos com brazil_ipca.csv**

## 1. Fundamentacao Teorica

### Holt-Winters Aditivo

$$\hat{y}_{t+h|t} = l_t + h b_t + s_{t+h-m(k+1)}$$

Equacoes de atualizacao:
$$l_t = \alpha(y_t - s_{t-m}) + (1-\alpha)(l_{t-1} + b_{t-1})$$
$$b_t = \beta(l_t - l_{t-1}) + (1-\beta) b_{t-1}$$
$$s_t = \gamma(y_t - l_{t-1} - b_{t-1}) + (1-\gamma) s_{t-m}$$

### Holt-Winters Multiplicativo

$$\hat{y}_{t+h|t} = (l_t + h b_t) \cdot s_{t+h-m(k+1)}$$

| Caracteristica | Aditivo | Multiplicativo |
|----------------|---------|----------------|
| Amplitude sazonal | Constante | Proporcional ao nivel |
| Dados com zeros | Funciona | Problematico |
| Transformacao log | Nao precisa | Equivalente a aditivo no log |

## 2. Setup e Carregamento dos Dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import json

from chronobox import HoltWinters, ETS
from scipy.stats import norm, probplot

np.random.seed(42)

# Diretorios
BASE_DIR = os.path.join(os.path.dirname(os.getcwd()))
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Carregar airline passengers
airline = pd.read_csv(os.path.join(DATA_DIR, 'airline.csv'), parse_dates=['date'])
airline.set_index('date', inplace=True)
y_airline = airline['passengers'].values.astype(float)

# Carregar IPCA
ipca = pd.read_csv(os.path.join(DATA_DIR, 'brazil_ipca.csv'), parse_dates=['date'])
ipca.set_index('date', inplace=True)
y_ipca = ipca['ipca'].values.astype(float)

print(f'Airline: {len(y_airline)} obs ({airline.index[0].strftime("%Y-%m")} a {airline.index[-1].strftime("%Y-%m")})')
print(f'IPCA: {len(y_ipca)} obs ({ipca.index[0].strftime("%Y-%m")} a {ipca.index[-1].strftime("%Y-%m")})')

## 3. Holt-Winters Aditivo no Airline

In [ ]:
# Ajustar Holt-Winters Aditivo no airline
hw_add = HoltWinters(seasonal='additive', seasonal_period=12)
res_add = hw_add.fit(y_airline)

print(res_add.summary())
print(f'\nParametros de suavizacao:')
print(f'  alpha (nivel): {res_add.alpha:.4f}')
print(f'  beta (tendencia): {res_add.beta:.4f}')
print(f'  gamma (sazonalidade): {res_add.gamma:.4f}')
print(f'  RMSE: {res_add.rmse:.2f}')

In [ ]:
# Decomposicao de componentes HW Aditivo
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

axes[0].plot(y_airline, label='Observado', color='steelblue')
axes[0].plot(res_add.fittedvalues, label='HW Aditivo', color='darkorange', linestyle='--')
axes[0].set_title('Holt-Winters Aditivo - Airline Passengers')
axes[0].legend()
axes[0].set_ylabel('Passageiros')

axes[1].plot(res_add.level, color='green')
axes[1].set_title('Nivel ($l_t$)')
axes[1].set_ylabel('Nivel')

axes[2].plot(res_add.trend, color='purple')
axes[2].set_title('Tendencia ($b_t$)')
axes[2].set_ylabel('Tendencia')

axes[3].plot(res_add.season, color='crimson')
axes[3].set_title('Sazonalidade ($s_t$)')
axes[3].set_ylabel('Sazonal')
axes[3].set_xlabel('Observacao')

plt.tight_layout()
plt.show()

## 4. Holt-Winters Multiplicativo no Airline

In [ ]:
# Ajustar Holt-Winters Multiplicativo no airline
hw_mul = HoltWinters(seasonal='multiplicative', seasonal_period=12)
res_mul = hw_mul.fit(y_airline)

print(res_mul.summary())
print(f'\nParametros de suavizacao:')
print(f'  alpha (nivel): {res_mul.alpha:.4f}')
print(f'  beta (tendencia): {res_mul.beta:.4f}')
print(f'  gamma (sazonalidade): {res_mul.gamma:.4f}')
print(f'  RMSE: {res_mul.rmse:.2f}')

## 5. Comparacao: Aditivo vs Multiplicativo

In [ ]:
# Comparacao aditivo vs multiplicativo
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(y_airline, label='Observado', color='steelblue', linewidth=1.5)
axes[0].plot(res_add.fittedvalues, label=f'HW Aditivo (RMSE={res_add.rmse:.2f})',
             color='darkorange', linestyle='--')
axes[0].plot(res_mul.fittedvalues, label=f'HW Multiplicativo (RMSE={res_mul.rmse:.2f})',
             color='green', linestyle='--')
axes[0].set_title('Comparacao: HW Aditivo vs Multiplicativo')
axes[0].legend()
axes[0].set_ylabel('Passageiros')

axes[1].plot(res_add.resid, label='Residuos Aditivo', color='darkorange', alpha=0.7)
axes[1].plot(res_mul.resid, label='Residuos Multiplicativo', color='green', alpha=0.7)
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Comparacao de Residuos')
axes[1].legend()
axes[1].set_ylabel('Residuo')
axes[1].set_xlabel('Observacao')

plt.tight_layout()
plt.show()

print(f'RMSE Aditivo:        {res_add.rmse:.4f}')
print(f'RMSE Multiplicativo: {res_mul.rmse:.4f}')
melhor = 'Multiplicativo' if res_mul.rmse < res_add.rmse else 'Aditivo'
print(f'\nMelhor modelo: HW {melhor}')

## 6. Damped Trend (Tendencia Amortecida)

A tendencia amortecida introduz um parametro $\phi \in (0,1)$ que **freia** a tendencia:

$$\hat{y}_{t+h|t} = l_t + (\phi + \phi^2 + \cdots + \phi^h) b_t + s_{t+h-m(k+1)}$$

Quando $h \to \infty$, a tendencia converge para $\frac{\phi}{1-\phi} b_t$.

In [ ]:
# Comparacao: tendencia linear vs amortecida
ets_aa = ETS(error='A', trend='A', seasonal='A', seasonal_period=12, damped=False)
res_aa = ets_aa.fit(y_airline)

ets_ad = ETS(error='A', trend='A', seasonal='A', seasonal_period=12, damped=True)
res_ad = ets_ad.fit(y_airline)

print(f'{"Modelo":<25} {"AIC":>10} {"BIC":>10} {"phi":>8}')
print('-' * 55)
print(f'{"ETS(A,A,A)":<25} {res_aa.aic:>10.2f} {res_aa.bic:>10.2f} {"  -":>8}')
phi_val = f'{res_ad.phi:.4f}' if res_ad.phi is not None else '  -'
print(f'{"ETS(A,Ad,A) damped":<25} {res_ad.aic:>10.2f} {res_ad.bic:>10.2f} {phi_val:>8}')

In [ ]:
# Impacto do damping nas previsoes
steps = 36
fc_linear = res_aa.forecast(steps=steps)
fc_damped = res_ad.forecast(steps=steps)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(range(len(y_airline)), y_airline, label='Observado', color='steelblue', linewidth=1.5)

idx_fc = range(len(y_airline), len(y_airline) + steps)
ax.plot(idx_fc, fc_linear, label='ETS(A,A,A) - Tendencia linear',
        color='darkorange', linewidth=2)
ax.plot(idx_fc, fc_damped, label='ETS(A,Ad,A) - Tendencia amortecida',
        color='green', linewidth=2, linestyle='--')

ax.axvline(len(y_airline) - 1, color='gray', linestyle=':', alpha=0.5)
ax.set_title('Impacto do Damping na Previsao (36 meses)')
ax.set_xlabel('Observacao')
ax.set_ylabel('Passageiros')
ax.legend()
plt.tight_layout()
plt.show()

print('A tendencia amortecida produz previsoes mais conservadoras.')
print('Para horizontes longos, o damping evita extrapolacoes excessivas.')

## 7. Previsao e Parametros

In [ ]:
# Previsao HW multiplicativo
fc_mul = res_mul.forecast(steps=24)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(range(len(y_airline)), y_airline, label='Observado', color='steelblue', linewidth=1.5)
ax.plot(range(len(y_airline)), res_mul.fittedvalues, label='HW Multiplicativo ajustado',
        color='darkorange', linestyle='--', alpha=0.7)

idx_fc = range(len(y_airline), len(y_airline) + 24)
ax.plot(idx_fc, fc_mul, label='Previsao HW Multiplicativo',
        color='crimson', linewidth=2)

ax.axvline(len(y_airline) - 1, color='gray', linestyle=':', alpha=0.5)
ax.set_title('Holt-Winters Multiplicativo - Previsao 24 Meses')
ax.set_xlabel('Observacao')
ax.set_ylabel('Passageiros')
ax.legend()
plt.tight_layout()
plt.show()

# Comparacao de parametros
print(f'{"Parametro":<12} {"HW Aditivo":>12} {"HW Multiplicativo":>18}')
print('-' * 45)
print(f'{"alpha":<12} {res_add.alpha:>12.4f} {res_mul.alpha:>18.4f}')
print(f'{"beta":<12} {res_add.beta:>12.4f} {res_mul.beta:>18.4f}')
print(f'{"gamma":<12} {res_add.gamma:>12.4f} {res_mul.gamma:>18.4f}')
print(f'{"RMSE":<12} {res_add.rmse:>12.4f} {res_mul.rmse:>18.4f}')

# Indices sazonais multiplicativos
print('\nIndices sazonais (HW Multiplicativo):')
meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
         'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
s_last = res_mul.season[-12:]
for m, v in zip(meses, s_last):
    bar = '#' * int(abs(v) * 30) if isinstance(v, float) else ''
    print(f'  {m}: {v:>8.4f} {bar}')

---

## Exercicio 1 - SOLUCAO: Holt-Winters no IPCA brasileiro

Usando o dataset `brazil_ipca.csv`:

1. Plote a serie do IPCA e identifique tendencia e sazonalidade
2. Ajuste HW aditivo e multiplicativo com `seasonal_period=12`
3. Compare RMSE e analise os residuos
4. Qual metodo e mais adequado para inflacao? Justifique.

**Dica:** O IPCA pode ter valores negativos (deflacao), o que torna o modelo multiplicativo problematico. Verifique isso!

In [ ]:
# Exercicio 1 - SOLUCAO
# Passo 1: Analise exploratoria do IPCA
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Serie temporal
axes[0, 0].plot(y_ipca, color='steelblue', linewidth=0.8)
axes[0, 0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0, 0].set_title('IPCA Mensal - Serie Completa')
axes[0, 0].set_ylabel('IPCA (%)')

# Histograma
axes[0, 1].hist(y_ipca, bins=30, color='steelblue', edgecolor='white')
axes[0, 1].axvline(0, color='red', linestyle='--')
axes[0, 1].set_title('Distribuicao do IPCA')
axes[0, 1].set_xlabel('IPCA (%)')

# Media mensal (sazonalidade)
ipca_monthly = np.array([y_ipca[i::12].mean() for i in range(12)])
axes[1, 0].bar(meses, ipca_monthly, color='steelblue', edgecolor='white')
axes[1, 0].set_title('Media Mensal do IPCA')
axes[1, 0].set_ylabel('IPCA medio (%)')
axes[1, 0].tick_params(axis='x', rotation=45)

# Amplitude sazonal por ano
n_anos_ipca = len(y_ipca) // 12
amp_ipca = [y_ipca[a*12:(a+1)*12].max() - y_ipca[a*12:(a+1)*12].min() for a in range(n_anos_ipca)]
niv_ipca = [y_ipca[a*12:(a+1)*12].mean() for a in range(n_anos_ipca)]
axes[1, 1].scatter(niv_ipca, amp_ipca, color='steelblue', s=40, edgecolors='black')
axes[1, 1].set_title('Amplitude Sazonal vs Nivel Medio')
axes[1, 1].set_xlabel('Nivel medio anual')
axes[1, 1].set_ylabel('Amplitude sazonal')

plt.tight_layout()
plt.show()

# Verificar presenca de valores negativos
n_negativos = np.sum(y_ipca < 0)
print(f'Valores negativos (deflacao): {n_negativos} de {len(y_ipca)} ({n_negativos/len(y_ipca)*100:.1f}%)')
print(f'Minimo: {y_ipca.min():.4f}')
print(f'Maximo: {y_ipca.max():.4f}')
print(f'Media: {y_ipca.mean():.4f}')

In [ ]:
# Passo 2: Ajustar HW aditivo e multiplicativo
# HW Aditivo
hw_ipca_add = HoltWinters(seasonal='additive', seasonal_period=12)
res_ipca_add = hw_ipca_add.fit(y_ipca)

print('=== HW Aditivo no IPCA ===')
print(res_ipca_add.summary())

# HW Multiplicativo - pode falhar com valores negativos
hw_multiplicativo_ok = True
try:
    # Para evitar problemas com valores negativos, deslocamos a serie
    # se houver negativos (o metodo multiplicativo requer todos positivos)
    if np.any(y_ipca <= 0):
        print('\nATENCAO: IPCA tem valores <= 0. HW Multiplicativo pode ser problematico.')
        # Tentamos com deslocamento
        shift = abs(y_ipca.min()) + 0.01
        y_ipca_shifted = y_ipca + shift
        hw_ipca_mul = HoltWinters(seasonal='multiplicative', seasonal_period=12)
        res_ipca_mul = hw_ipca_mul.fit(y_ipca_shifted)
        print(f'Aplicado deslocamento de {shift:.4f} para tornar serie positiva.')
    else:
        hw_ipca_mul = HoltWinters(seasonal='multiplicative', seasonal_period=12)
        res_ipca_mul = hw_ipca_mul.fit(y_ipca)
    
    print('\n=== HW Multiplicativo no IPCA ===')
    print(res_ipca_mul.summary())
except Exception as e:
    print(f'\nHW Multiplicativo FALHOU: {e}')
    hw_multiplicativo_ok = False

In [ ]:
# Passo 3: Comparacao de RMSE e residuos
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Ajuste aditivo
axes[0, 0].plot(y_ipca, label='IPCA', color='steelblue', linewidth=0.8)
axes[0, 0].plot(res_ipca_add.fittedvalues, label=f'HW Aditivo (RMSE={res_ipca_add.rmse:.4f})',
                color='darkorange', linestyle='--', alpha=0.8)
axes[0, 0].set_title('HW Aditivo - Ajuste')
axes[0, 0].legend(fontsize=9)
axes[0, 0].set_ylabel('IPCA (%)')

# Residuos aditivo
axes[0, 1].plot(res_ipca_add.resid, color='darkorange', linewidth=0.8)
axes[0, 1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0, 1].set_title('HW Aditivo - Residuos')
axes[0, 1].set_ylabel('Residuo')

if hw_multiplicativo_ok:
    # Ajuste multiplicativo (usando serie deslocada se necessario)
    y_plot_mul = y_ipca_shifted if np.any(y_ipca <= 0) else y_ipca
    axes[1, 0].plot(y_plot_mul, label='IPCA (deslocado)' if np.any(y_ipca <= 0) else 'IPCA',
                    color='steelblue', linewidth=0.8)
    axes[1, 0].plot(res_ipca_mul.fittedvalues, label=f'HW Multiplicativo (RMSE={res_ipca_mul.rmse:.4f})',
                    color='green', linestyle='--', alpha=0.8)
    axes[1, 0].set_title('HW Multiplicativo - Ajuste')
    axes[1, 0].legend(fontsize=9)
    axes[1, 0].set_ylabel('IPCA (%)')
    
    axes[1, 1].plot(res_ipca_mul.resid, color='green', linewidth=0.8)
    axes[1, 1].axhline(0, color='red', linestyle='--', alpha=0.5)
    axes[1, 1].set_title('HW Multiplicativo - Residuos')
    axes[1, 1].set_ylabel('Residuo')

plt.tight_layout()
plt.show()

# Tabela comparativa
print(f'{"Metrica":<15} {"HW Aditivo":>15}', end='')
if hw_multiplicativo_ok:
    print(f' {"HW Multiplicativo":>18}')
else:
    print()
print('-' * 50)

# Metricas aditivo
r_add = res_ipca_add.resid
rmse_add = res_ipca_add.rmse
mae_add = np.mean(np.abs(r_add))
# MAPE: cuidado com zeros no IPCA
mask_nz = np.abs(y_ipca) > 0.001
mape_add = np.mean(np.abs(r_add[mask_nz] / y_ipca[mask_nz])) * 100

print(f'{"RMSE":<15} {rmse_add:>15.4f}', end='')
if hw_multiplicativo_ok:
    print(f' {res_ipca_mul.rmse:>18.4f}')
else:
    print()
print(f'{"MAE":<15} {mae_add:>15.4f}', end='')
if hw_multiplicativo_ok:
    r_mul = res_ipca_mul.resid
    mae_mul = np.mean(np.abs(r_mul))
    print(f' {mae_mul:>18.4f}')
else:
    print()
print(f'{"MAPE (%)":<15} {mape_add:>15.4f}')

In [ ]:
# Passo 4: Diagnostico de residuos HW Aditivo no IPCA
resid_ipca = res_ipca_add.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(resid_ipca, color='darkorange', linewidth=0.8)
axes[0, 0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0, 0].set_title('Residuos ao Longo do Tempo')
axes[0, 0].set_ylabel('Residuo')

axes[0, 1].hist(resid_ipca, bins=25, color='darkorange', edgecolor='white', density=True)
x_h = np.linspace(resid_ipca.min(), resid_ipca.max(), 100)
axes[0, 1].plot(x_h, norm.pdf(x_h, resid_ipca.mean(), resid_ipca.std()), 'r-', linewidth=2)
axes[0, 1].set_title('Histograma dos Residuos')

n_r = len(resid_ipca)
acf_ipca = np.array([np.corrcoef(resid_ipca[:n_r-k], resid_ipca[k:])[0, 1] for k in range(25)])
acf_ipca[0] = 1.0
ci_ipca = 1.96 / np.sqrt(n_r)
axes[1, 0].bar(range(25), acf_ipca, width=0.3, color='darkorange')
axes[1, 0].axhline(ci_ipca, color='red', linestyle='--', alpha=0.7)
axes[1, 0].axhline(-ci_ipca, color='red', linestyle='--', alpha=0.7)
axes[1, 0].set_title('ACF dos Residuos')

probplot(resid_ipca, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot')

plt.suptitle('Diagnostico de Residuos - HW Aditivo (IPCA)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Conclusao do Exercicio 1

Para o IPCA brasileiro, o **HW Aditivo** e mais adequado porque:

1. **Valores negativos**: O IPCA pode ter valores negativos (periodos de deflacao), o que torna o modelo multiplicativo problematico ou requer deslocamento artificial da serie.
2. **Amplitude sazonal relativamente constante**: A sazonalidade do IPCA nao varia proporcionalmente ao nivel.
3. **Simplicidade**: O HW Aditivo nao requer transformacoes adicionais.
4. **RMSE competitivo**: O modelo aditivo apresenta bom ajuste sem necessidade de manipulacao dos dados.

---

## Exercicio 2 - SOLUCAO: Damped Trend no IPCA

1. Ajuste ETS(A,A,A) e ETS(A,Ad,A) no IPCA com `seasonal_period=12`
2. Faca previsoes de 12 e 36 meses com cada modelo
3. Plote as previsoes lado a lado
4. Qual modelo e mais prudente para previsoes de inflacao? Por que?
5. Qual o valor estimado de $\phi$? O que ele indica?

In [ ]:
# Exercicio 2 - SOLUCAO
# Passo 1: Ajustar ETS(A,A,A) e ETS(A,Ad,A) no IPCA
ets_aaa_ipca = ETS(error='A', trend='A', seasonal='A', seasonal_period=12)
res_aaa_ipca = ets_aaa_ipca.fit(y_ipca)

ets_ada_ipca = ETS(error='A', trend='A', seasonal='A', seasonal_period=12, damped=True)
res_ada_ipca = ets_ada_ipca.fit(y_ipca)

print(f'{"Modelo":<20} {"alpha":>8} {"beta":>8} {"gamma":>8} {"phi":>8} {"AIC":>10} {"BIC":>10}')
print('-' * 75)
print(f'{"ETS(A,A,A)":<20} {res_aaa_ipca.alpha:>8.4f} {res_aaa_ipca.beta:>8.4f} {res_aaa_ipca.gamma:>8.4f} {"   -":>8} {res_aaa_ipca.aic:>10.2f} {res_aaa_ipca.bic:>10.2f}')
phi_ipca = res_ada_ipca.phi if res_ada_ipca.phi is not None else 0
print(f'{"ETS(A,Ad,A)":<20} {res_ada_ipca.alpha:>8.4f} {res_ada_ipca.beta:>8.4f} {res_ada_ipca.gamma:>8.4f} {phi_ipca:>8.4f} {res_ada_ipca.aic:>10.2f} {res_ada_ipca.bic:>10.2f}')

# Metricas
print(f'\n{"Modelo":<20} {"RMSE":>10} {"MAE":>10} {"MAPE(%)":>10}')
print('-' * 55)
for nome, res in [('ETS(A,A,A)', res_aaa_ipca), ('ETS(A,Ad,A)', res_ada_ipca)]:
    r = res.resid
    rmse_v = np.sqrt(np.mean(r**2))
    mae_v = np.mean(np.abs(r))
    mask_nz = np.abs(y_ipca) > 0.001
    mape_v = np.mean(np.abs(r[mask_nz] / y_ipca[mask_nz])) * 100
    print(f'{nome:<20} {rmse_v:>10.4f} {mae_v:>10.4f} {mape_v:>10.2f}')

In [ ]:
# Passo 2 e 3: Previsoes de 12 e 36 meses
fc_aaa_12 = res_aaa_ipca.forecast(steps=12)
fc_aaa_36 = res_aaa_ipca.forecast(steps=36)
fc_ada_12 = res_ada_ipca.forecast(steps=12)
fc_ada_36 = res_ada_ipca.forecast(steps=36)

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Previsao 12 meses
n_show = 60  # Mostrar ultimos 60 obs + previsao
axes[0].plot(range(len(y_ipca)-n_show, len(y_ipca)), y_ipca[-n_show:],
             label='Observado', color='steelblue', linewidth=1.5)
idx_12 = range(len(y_ipca), len(y_ipca) + 12)
axes[0].plot(idx_12, fc_aaa_12, label='ETS(A,A,A)', color='darkorange', linewidth=2)
axes[0].plot(idx_12, fc_ada_12, label='ETS(A,Ad,A) damped', color='green', linewidth=2, linestyle='--')
axes[0].axvline(len(y_ipca) - 1, color='gray', linestyle=':', alpha=0.5)
axes[0].set_title('Previsao IPCA - 12 Meses')
axes[0].set_ylabel('IPCA (%)')
axes[0].legend()

# Previsao 36 meses
axes[1].plot(range(len(y_ipca)-n_show, len(y_ipca)), y_ipca[-n_show:],
             label='Observado', color='steelblue', linewidth=1.5)
idx_36 = range(len(y_ipca), len(y_ipca) + 36)
axes[1].plot(idx_36, fc_aaa_36, label='ETS(A,A,A)', color='darkorange', linewidth=2)
axes[1].plot(idx_36, fc_ada_36, label='ETS(A,Ad,A) damped', color='green', linewidth=2, linestyle='--')
axes[1].axvline(len(y_ipca) - 1, color='gray', linestyle=':', alpha=0.5)
axes[1].set_title('Previsao IPCA - 36 Meses')
axes[1].set_ylabel('IPCA (%)')
axes[1].set_xlabel('Observacao')
axes[1].legend()

plt.tight_layout()
plt.show()

# Comparacao das previsoes
print(f'{"Horizonte":<12} {"ETS(A,A,A)":>12} {"ETS(A,Ad,A)":>12} {"Diferenca":>12}')
print('-' * 50)
for h in [1, 6, 12, 24, 36]:
    v_lin = fc_aaa_36[h-1] if h <= 36 else float('nan')
    v_dmp = fc_ada_36[h-1] if h <= 36 else float('nan')
    print(f'{f"h={h}":<12} {v_lin:>12.4f} {v_dmp:>12.4f} {abs(v_lin - v_dmp):>12.4f}')

In [ ]:
# Passo 4 e 5: Interpretacao de phi e conclusao
print('=' * 60)
print('ANALISE: Damped Trend para Previsao de Inflacao')
print('=' * 60)

phi_est = res_ada_ipca.phi if res_ada_ipca.phi is not None else 0
print(f'\nParametro phi estimado: {phi_est:.4f}')
print(f'\nInterpretacao de phi = {phi_est:.4f}:')
if phi_est > 0.98:
    print('  -> phi proximo de 1: tendencia quase linear (damping minimo)')
elif phi_est > 0.9:
    print('  -> phi alto: tendencia levemente amortecida')
elif phi_est > 0.8:
    print('  -> phi moderado: tendencia amortecida significativamente')
else:
    print('  -> phi baixo: tendencia fortemente amortecida')

print(f'\nConvergencia da tendencia:')
b_est = res_ada_ipca.b0 if res_ada_ipca.b0 is not None else 0
if phi_est < 1:
    tendencia_lp = phi_est / (1 - phi_est) * b_est
    print(f'  Tendencia inicial (b0): {b_est:.4f}')
    print(f'  Tendencia acumulada longo prazo: phi/(1-phi)*b0 = {tendencia_lp:.4f}')

print(f'\nConclusao:')
print('  O modelo ETS(A,Ad,A) com tendencia amortecida e MAIS PRUDENTE para')
print('  previsoes de inflacao porque:')
print('  1. Evita extrapolar tendencias indefinidamente')
print('  2. Reflete a expectativa de que bancos centrais atuam para estabilizar inflacao')
print('  3. Previsoes de longo prazo convergem para um nivel, nao crescem sem limite')
print('  4. Mais conservador = menor risco de previsoes absurdas')

### Conclusao do Exercicio 2

O modelo **ETS(A,Ad,A)** com tendencia amortecida e mais prudente para inflacao:

1. **Previsoes convergentes**: A tendencia nao extrapola indefinidamente
2. **Consistente com politica monetaria**: Bancos centrais atuam para estabilizar inflacao
3. **Diferenca cresce com o horizonte**: Para h=12, as previsoes sao similares; para h=36, a diferenca e significativa
4. **Parametro phi**: Indica a taxa de amortecimento da tendencia

---

## 8. Salvando Resultados para Comparacao

In [ ]:
# Salvar resultados HW em outputs/hw_results.json
hw_results = {
    'airline_hw_additive': {
        'alpha': round(res_add.alpha, 6),
        'beta': round(res_add.beta, 6),
        'gamma': round(res_add.gamma, 6),
        'rmse': round(float(res_add.rmse), 4),
        'mse': round(float(res_add.mse), 4),
        'mae': round(float(np.mean(np.abs(res_add.resid))), 4),
        'mape': round(float(np.mean(np.abs(res_add.resid / y_airline)) * 100), 4),
        'l0': round(float(res_add.l0), 4),
        'b0': round(float(res_add.b0), 4),
    },
    'airline_hw_multiplicative': {
        'alpha': round(res_mul.alpha, 6),
        'beta': round(res_mul.beta, 6),
        'gamma': round(res_mul.gamma, 6),
        'rmse': round(float(res_mul.rmse), 4),
        'mse': round(float(res_mul.mse), 4),
        'mae': round(float(np.mean(np.abs(res_mul.resid))), 4),
        'mape': round(float(np.mean(np.abs(res_mul.resid / y_airline)) * 100), 4),
        'l0': round(float(res_mul.l0), 4),
        'b0': round(float(res_mul.b0), 4),
    },
    'ipca_hw_additive': {
        'alpha': round(res_ipca_add.alpha, 6),
        'beta': round(res_ipca_add.beta, 6),
        'gamma': round(res_ipca_add.gamma, 6),
        'rmse': round(float(res_ipca_add.rmse), 4),
        'mse': round(float(res_ipca_add.mse), 4),
        'mae': round(float(np.mean(np.abs(res_ipca_add.resid))), 4),
        'l0': round(float(res_ipca_add.l0), 4),
        'b0': round(float(res_ipca_add.b0), 4),
    },
    'ipca_ets_aaa': {
        'alpha': round(res_aaa_ipca.alpha, 6),
        'beta': round(res_aaa_ipca.beta, 6) if res_aaa_ipca.beta is not None else None,
        'gamma': round(res_aaa_ipca.gamma, 6) if res_aaa_ipca.gamma is not None else None,
        'aic': round(res_aaa_ipca.aic, 4),
        'bic': round(res_aaa_ipca.bic, 4),
        'aicc': round(res_aaa_ipca.aicc, 4),
        'loglike': round(res_aaa_ipca.loglik, 4),
        'rmse': round(float(np.sqrt(np.mean(res_aaa_ipca.resid**2))), 4),
        'mae': round(float(np.mean(np.abs(res_aaa_ipca.resid))), 4),
    },
    'ipca_ets_ada': {
        'alpha': round(res_ada_ipca.alpha, 6),
        'beta': round(res_ada_ipca.beta, 6) if res_ada_ipca.beta is not None else None,
        'gamma': round(res_ada_ipca.gamma, 6) if res_ada_ipca.gamma is not None else None,
        'phi': round(float(res_ada_ipca.phi), 6) if res_ada_ipca.phi is not None else None,
        'aic': round(res_ada_ipca.aic, 4),
        'bic': round(res_ada_ipca.bic, 4),
        'aicc': round(res_ada_ipca.aicc, 4),
        'loglike': round(res_ada_ipca.loglik, 4),
        'rmse': round(float(np.sqrt(np.mean(res_ada_ipca.resid**2))), 4),
        'mae': round(float(np.mean(np.abs(res_ada_ipca.resid))), 4),
    },
}

output_path = os.path.join(OUTPUT_DIR, 'hw_results.json')
with open(output_path, 'w') as f:
    json.dump(hw_results, f, indent=2)

print(f'Resultados HW salvos em: {output_path}')
print(f'Total de configuracoes salvas: {len(hw_results)}')
print('\nResumo:')
for key in hw_results:
    rmse_v = hw_results[key].get('rmse', 'N/A')
    print(f'  {key}: RMSE={rmse_v}')

---

## Conclusao

Neste notebook aprendemos:

1. **HW Aditivo vs Multiplicativo**: Escolher com base na natureza da sazonalidade
2. **Damped Trend**: Parametro phi controla o amortecimento da tendencia
3. **Cuidados com dados reais**: Valores negativos impedem uso direto do multiplicativo
4. **Prudencia em previsoes de inflacao**: Tendencia amortecida evita extrapolacoes excessivas
5. **Diagnostico**: Sempre verificar residuos antes de confiar nas previsoes

**Proximo notebook:** Auto-ETS, Theta Method e combinacao de previsoes